[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/13PkpkDbjLCrttyEV42UBihobtuiKDAO-/view?usp=drive_link)

# LLM Evaluation – Partial Dataset

This notebook demonstrates how to evaluate when you have only questions (no answers yet). Floeval generates `llm_response` for each sample at runtime, then runs the metrics on the generated responses.

**Objectives**
- Install Floeval and configure credentials
- Build a partial dataset (samples without `llm_response`)
- Set `dataset_generator_model` so Floeval generates responses
- Run evaluation and inspect results

## 1. Installation

Install Floeval before running this notebook.

In [ ]:
%pip install git+https://github.com/FloTorch/floeval.git@dev

## 2. Configuration Constants

Set the following constants before running. Replace placeholder values with your API credentials and model identifiers.

**Provider flexibility:** You can use any OpenAI-compatible provider (OpenAI, Azure OpenAI, Anthropic, local models, etc.) — set the appropriate `base_url` and model names for your provider.

**Using FloTorch:** If you want to use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass
# LLM and API configuration

OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_API_KEY = getpass.getpass("your-api-key")
OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

## 3. Imports

The following cell imports the evaluation components and the LLM configuration schema.

In [ ]:
from floeval import Evaluation, DatasetLoader
from floeval.config.schemas.io.llm import OpenAIProviderConfig

## 4. Configure the LLM

The LLM configuration is built using the constants defined above. It is used for both response generation and evaluation.

In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    chat_model=OPENAI_CHAT_MODEL,
    embedding_model=OPENAI_EMBEDDING_MODEL,
)

## 5. Load the Partial Dataset

A partial dataset is created with `user_input` only. The `partial_dataset=True` flag indicates that `llm_response` will be generated by Floeval at runtime.

In [ ]:
partial_dataset = DatasetLoader.from_samples(
    [
        {"user_input": "What is Python?"},
        {"user_input": "What is RAG?"},
    ],
    partial_dataset=True,
)
print(f"Partial dataset loaded: {len(partial_dataset.samples)} samples (no llm_response)")

## 6. Create and Run the Evaluation

The `dataset_generator_model` parameter specifies which model Floeval uses to generate responses. For each sample, Floeval sends `user_input` to the LLM, stores the response, and then runs the metrics on the generated output.

In [ ]:
evaluation = Evaluation(
    dataset=partial_dataset,
    llm_config=llm_config,
    metrics=["answer_relevancy"],
    default_provider="ragas",
    dataset_generator_model=OPENAI_CHAT_MODEL,
)

results = evaluation.run()
print("Aggregate scores:", results.aggregate_scores)

## 7. Inspect Generated Responses

Each sample result includes the generated `llm_response` and the metric scores. This enables inspection of both the generated text and the evaluation scores per sample.

In [ ]:
for i, sr in enumerate(results.sample_results, start=1):
    print(f"Sample {i}: {sr['user_input']}")
    print(f"  Generated: {sr.get('llm_response', '')[:80]}...")
    for key, data in sr.get("metrics", {}).items():
        print(f"  {key}: {data.get('score')}")

## Summary

This notebook demonstrated the evaluation of LLM responses using a partial dataset with Floeval.

The key components included:

1. **Partial Dataset Loading**: A dataset with `user_input` only was loaded using `partial_dataset=True`.
2. **LLM Configuration**: The OpenAI-compatible provider was configured for both generation and evaluation.
3. **Response Generation**: The `dataset_generator_model` parameter enabled Floeval to generate responses at runtime before scoring.
4. **Evaluation Execution**: The `answer_relevancy` metric was run on the generated responses.
5. **Results Inspection**: Generated text and per-sample scores were accessed through `results.sample_results`.

This example showcases the workflow for evaluating when you have questions but no pre-generated answers.